In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [3]:
# -----------------------------
# 1. 4-bit configuration
# -----------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [4]:
# -----------------------------
# 2. Load tokenizer
# -----------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [5]:
# -----------------------------
# 3. Load Qwen
# -----------------------------

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [6]:
# -----------------------------
# 4. Prepare for QLoRA
# -----------------------------

model = prepare_model_for_kbit_training(model)

In [7]:
# -----------------------------
# 5. LoRA configuration
# -----------------------------

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

In [8]:
# -----------------------------
# 6. Attach LoRA
# -----------------------------

model = get_peft_model(
    model,
    lora_config
)


In [9]:
# -----------------------------
# 7. Print trainable parameters
# -----------------------------

model.print_trainable_parameters()

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
